# 17 - Node2Vec Embeddings

## Overview

This notebook learns numerical representations (embeddings) for jobs and skills from the job–skill bipartite graph constructed in Notebook 16, which forms the structural foundation of Chapter 2. In that graph, jobs and skills are represented as distinct node types, with weighted edges encoding the strength of each job–skill relationship. This representation captures the relational structure of the job market explicitly, but remains symbolic in nature and is not directly amenable to tasks such as clustering, similarity analysis, or optimisation.

Node2Vec addresses this limitation by converting the graph structure into a continuous vector space. Using weighted random walks over the bipartite graph, Node2Vec samples local and global neighbourhood structure and learns fixed-length embeddings for every job and skill. Nodes that occupy similar positions in the graph—jobs requiring similar bundles of skills, or skills that co-occur across similar sets of jobs—are embedded close together in the learned space. In this way, the high-dimensional relational information encoded in the graph is compressed into a geometry that preserves meaningful structural similarity.

The resulting embeddings provide the numerical substrate for the remainder of Chapter 2. They make it possible to cluster jobs into latent job families, analyse skill ecosystems, and quantify proximity between roles in a way that reflects the underlying structure of the job market rather than surface-level features. This module is therefore purely representational: it does not perform clustering or interpretation itself, but transforms the validated graph into reusable, geometry-aware embeddings that enable all downstream analyses.



## Set up

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
#===
import sys
from pathlib import Path
#===
from node2vec import Node2Vec


In [2]:
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

PosixPath('/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine')

In [3]:
from src.job_intel.features.graph_job_skill import build_job_skill_bipartite_graph

## Build graph

In [4]:
G, mat, thres, df = build_job_skill_bipartite_graph(threshold=0.5, save_graph_pickle=False)

✅ Building the job probability matrix.
✅ Job probability matrix built.
✅ Creating graph...
✅ Adding job nodes...
✅ Adding skill nodes...
✅ Nodes added successfully.
✅ Adding edges using threshold = 0.5.
✅ Edges added. Total edges = 40313
✅ Graph successfully built!


## Embeddings

In [5]:
node = Node2Vec(G,
                dimensions= 64,
                walk_length=20,
                num_walks=40,
                workers=4,
                weight_key='weight',
                seed = 42)

Computing transition probabilities:   0%|          | 0/6188 [00:00<?, ?it/s]

Generating walks (CPU: 4): 100%|██████████| 10/10 [01:26<00:00,  8.69s/it]


In [6]:
model = node.fit(window=10, min_count=1, batch_words=4)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [7]:
model2 = node.fit(window=10, min_count=1, batch_words=4)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


### Checks

In [8]:
len(model.wv) == G.number_of_nodes()

True

In [9]:
model.wv.vector_size == 64

True

In [10]:
model.wv.key_to_index

{'soft_skills__core_prob': 0,
 'analytics_stats__basic_prob': 1,
 'soft_skills__leadership_prob': 2,
 'core_programming__basic_prob': 3,
 'db_storage__basic_prob': 4,
 'bi_viz__basic_prob': 5,
 'data_engineering_pipelines__intermediate_prob': 6,
 'bi_viz__intermediate_prob': 7,
 'domain_specific__none_prob': 8,
 'db_storage__intermediate_prob': 9,
 'cloud__basic_prob': 10,
 'ml_ai__basic_prob': 11,
 'ml_ai__intermediate_prob': 12,
 'data_engineering_pipelines__advanced_prob': 13,
 'productivity_workflow__intermediate_prob': 14,
 'productivity_workflow__basic_prob': 15,
 'analytics_stats__intermediate_prob': 16,
 'core_programming__intermediate_prob': 17,
 'cloud__intermediate_prob': 18,
 'ml_ai__advanced_prob': 19,
 'db_storage__advanced_prob': 20,
 'cloud__advanced_prob': 21,
 'data_engineering_pipelines__basic_prob': 22,
 'analytics_stats__advanced_prob': 23,
 'bi_viz__advanced_prob': 24,
 'productivity_workflow__advanced_prob': 25,
 '268': 26,
 '3472': 27,
 '1250': 28,
 '915': 29,
 

In [11]:
def top_neighbors(wv, node, topn=10):
    return [k for k, _ in wv.most_similar(node, topn=topn)]

# choose anchors (2 jobs + 2 skills)
anchors = [0, 1, "core_programming__basic_prob", "data_engineering_pipelines__intermediate_prob"]

# IMPORTANT: keys in gensim are often strings
anchors = [str(a) for a in anchors]

nbrs_1 = {a: top_neighbors(model.wv, a, topn=10) for a in anchors}

# --- re-fit once (same params) to test stability ---
# model2 = node2vec.fit(window=10, min_count=1, batch_words=4)  # run this in your notebook
# after you have model2:

nbrs_2 = {a: top_neighbors(model2.wv, a, topn=10) for a in anchors}

for a in anchors:
    overlap = len(set(nbrs_1[a]).intersection(nbrs_2[a]))
    print(a, "overlap_top10 =", overlap)



0 overlap_top10 = 2
1 overlap_top10 = 4
core_programming__basic_prob overlap_top10 = 7
data_engineering_pipelines__intermediate_prob overlap_top10 = 9


### Extract and save embeddings

In [ ]:
job_nodes = [n for n, d in G.nodes(data=True) if d.get("bipartite") == "job"]
skill_nodes = [n for n, d in G.nodes(data=True) if d.get("bipartite") == "skill"]

In [ ]:
dim = model.wv.vector_size
cols = [f"emb_{i}" for i in range(dim)]

In [ ]:
cols

In [ ]:
job_emb = pd.DataFrame([model.wv[str(n)] for n in job_nodes], index=job_nodes, columns=cols)
skill_emb = pd.DataFrame([model.wv[str(n)] for n in skill_nodes], index=skill_nodes, columns=cols)
job_emb.index.name = "job_id"
skill_emb.index.name = "skill"

In [ ]:
from src.job_intel.config import PROCESSED_DATA_DIR
job_emb.to_csv(PROCESSED_DATA_DIR / "job_embeddings_node2vec_v01.csv")
skill_emb.to_csv(PROCESSED_DATA_DIR / "skill_embeddings_node2vec_v01.csv")


# == End of Notebook == 